In [44]:
import warnings
warnings.filterwarnings("ignore")

In [45]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [46]:
data = pd.read_csv("../datasets/final_gemini_extract.csv")

In [47]:
df = data.copy()

In [48]:
from sklearn.preprocessing import MultiLabelBinarizer

# Step 1.1: Split keywords (comma-separated) into lists
df['Search_Keywords_List'] = df['Search_Keywords'].apply(lambda x: [kw.strip().lower() for kw in x.split(',') if kw.strip()])

# Step 1.2: Fit MultiLabelBinarizer
mlb = MultiLabelBinarizer()
keyword_binarized = mlb.fit_transform(df['Search_Keywords_List'])

# Step 1.3: Create a DataFrame from binarized keywords
keyword_df = pd.DataFrame(keyword_binarized, columns=[f"kw_{kw}" for kw in mlb.classes_])

# Step 1.4: Reset index to align
keyword_df.index = df.index

# Step 1.5: Concatenate back to original DataFrame
df = pd.concat([df, keyword_df], axis=1)

print(" Keywords binarized and added as features")
print("Total keyword features:", keyword_df.shape[1])


 Keywords binarized and added as features
Total keyword features: 11149


In [49]:
# Step 2.1: Define float (numeric) columns
float_cols = ['min_age', 'max_age', 'annual_income', 'max_income']

# Step 2.2: Define original binary eligibility columns (targets)
target_cols = [col for col in df.columns if (
    col.startswith("is_") or 
    col.startswith("has_") or 
    col.startswith("belongs_to_") or 
    col.startswith("receives_") or 
    col.startswith("lives_in_") or
    col.startswith("no")
)]

# Step 2: Convert binary object cols to 0/1 (yes/true/1 → 1; else → 0)
for col in target_cols:
    df[col] = df[col].fillna(False).astype(str).str.lower().isin(['1', 'true', 'yes']).astype(int)

# Fill missing numeric values with -1 (or median if you prefer)
df[float_cols] = df[float_cols].fillna(-1)

# Step 2.3: Define keyword features (from previous step)
keyword_cols = [col for col in df.columns if col.startswith("kw_")]

# Step 2.4: Final input columns = binary eligibility + float + keyword features
input_cols = float_cols + keyword_cols   # target cols used as input features too

# Step 2.5: Create X and y
X = df[input_cols].copy()
y = df[target_cols].copy()


In [50]:
from sklearn.preprocessing import MinMaxScaler

# Initialize scaler
scaler = MinMaxScaler()

# Fit-transform the float columns
X[float_cols] = scaler.fit_transform(X[float_cols])


In [51]:
# Just to be sure all columns are 0/1 integers
y = y.astype(int)

In [52]:
from sklearn.model_selection import train_test_split

In [53]:
# Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [54]:
# Rename y_train columns to avoid duplicates with X_train
y_train_renamed = y_train.copy()
y_train_renamed.columns = ['label_' + col for col in y_train.columns]

# Merge for resampling
train_df = pd.concat([X_train, y_train_renamed], axis=1)


In [13]:
from sklearn.utils import resample

# Define threshold to call a label "minority"
minority_threshold = 50

# Store resampled dataframes
resampled_dfs = []

for label in y_train_renamed.columns:
    count = train_df[label].sum()
    
    if count < minority_threshold:
        df_minority = train_df[train_df[label] == 1]
        df_majority = train_df[train_df[label] == 0]

        df_minority_upsampled = resample(
            df_minority,
            replace=True,
            n_samples=len(df_majority),
            random_state=42
        )

        balanced_df = pd.concat([df_majority, df_minority_upsampled])
        resampled_dfs.append(balanced_df)
        print(f"Oversampled: {label} from {int(count)} to {len(df_majority)}")
    else:
        resampled_dfs.append(train_df)

# Combine and shuffle
train_upsampled = pd.concat(resampled_dfs).drop_duplicates()
train_upsampled = train_upsampled.sample(frac=1, random_state=42)

# Separate features and labels again
X_train_balanced = train_upsampled[X_train.columns]
y_train_balanced = train_upsampled[y_train_renamed.columns]
y_train_balanced.columns = [col.replace("label_", "") for col in y_train_balanced.columns]


Oversampled: label_is_transgender from 40 to 2920
Oversampled: label_is_job_seeker from 22 to 2938
Oversampled: label_is_dropout from 5 to 2955
Oversampled: label_is_migrant_worker from 12 to 2948
Oversampled: label_has_large_family from 1 to 2959
Oversampled: label_no_asset_ownership from 15 to 2945
Oversampled: label_receives_pension from 49 to 2911
Oversampled: label_is_orphan from 36 to 2924
Oversampled: label_is_child_of_single_parent from 21 to 2939
Oversampled: label_is_woman_headed_household from 10 to 2950
Oversampled: label_has_disabled_family_member from 31 to 2929
Oversampled: label_lives_in_slum from 1 to 2959
Oversampled: label_is_homeless from 16 to 2944
Oversampled: label_has_no_house from 24 to 2936
Oversampled: label_has_pukka_house from 2 to 2958
Oversampled: label_has_kutcha_house from 10 to 2950
Oversampled: label_belongs_to_weaver_community from 46 to 2914
Oversampled: label_is_person_with_hiv from 9 to 2951
Oversampled: label_is_victim_of_abuse from 21 to 2939
Ov

In [14]:
from xgboost import XGBClassifier
from sklearn.multioutput import MultiOutputClassifier

In [15]:
# Define base XGBoost model
xgb = XGBClassifier(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    use_label_encoder=False,
    eval_metric='logloss',
    random_state=42
)

# Wrap with MultiOutputClassifier
clf1 = MultiOutputClassifier(xgb, n_jobs=1)

# Convert to NumPy before training (final fix)
clf1.fit(X_train_balanced.to_numpy(), y_train_balanced.to_numpy())

MultiOutputClassifier(estimator=XGBClassifier(base_score=None, booster=None,
                                              callbacks=None,
                                              colsample_bylevel=None,
                                              colsample_bynode=None,
                                              colsample_bytree=0.8, device=None,
                                              early_stopping_rounds=None,
                                              enable_categorical=False,
                                              eval_metric='logloss',
                                              feature_types=None, gamma=None,
                                              grow_policy=None,
                                              importance_type=None,
                                              interaction_constraints=None,
                                              learning_rate=0.1, max_bin=None,
                                              max_cat_threshold=None,
                                              max_cat_to_onehot=None,
                                              max_delta_step=None, max_depth=5,
                                              max_leaves=None,
                                              min_child_weight=None,
                                              missing=nan,
                                              monotone_constraints=None,
                                              multi_strategy=None,
                                              n_estimators=100, n_jobs=None,
                                              num_parallel_tree=None,
                                              random_state=42, ...),
                      n_jobs=1)

In [55]:
from sklearn.metrics import f1_score, accuracy_score

def evaluate_model(X, y_true, label="", col_names=None):
    y_pred = clf1.predict(X)

    # Overall metrics
    overall_f1 = f1_score(y_true, y_pred, average='samples', zero_division=0)
    overall_acc = accuracy_score(y_true, y_pred)

    print(f"\n{label} Set Evaluation")
    print("Overall Sample-wise F1 Score: ", round(overall_f1, 3))
    print("Overall Sample-wise Accuracy:", round(overall_acc, 3))

    # Label-wise metrics
    print(f"\n{label} Set - First 5 Labels:")
    for i, col in enumerate(col_names[:5]):
        f1 = f1_score(y_true[:, i], y_pred[:, i], zero_division=0)
        acc = accuracy_score(y_true[:, i], y_pred[:, i])
        print(f"{col}: F1 = {round(f1, 3)}, Acc = {round(acc, 3)}")

    return overall_f1, overall_acc


In [56]:
from sklearn.metrics import f1_score, accuracy_score

def evaluate_model(X, y_true, label="", col_names=None):
    y_pred = clf1.predict(X)

    # Overall metrics
    overall_f1 = f1_score(y_true, y_pred, average='samples', zero_division=0)
    overall_acc = accuracy_score(y_true, y_pred)

    print(f"\n{label} Set Evaluation")
    print("Overall Sample-wise F1 Score: ", round(overall_f1, 3))
    print("Overall Sample-wise Accuracy:", round(overall_acc, 3))

    # Label-wise metrics
    print(f"\n{label} Set - First 5 Labels:")
    for i, col in enumerate(col_names[:5]):
        f1 = f1_score(y_true[:, i], y_pred[:, i], zero_division=0)
        acc = accuracy_score(y_true[:, i], y_pred[:, i])
        print(f"{col}: F1 = {round(f1, 3)}, Acc = {round(acc, 3)}")

    return overall_f1, overall_acc


In [57]:
# Extract column names before converting to numpy
label_columns = y_train_balanced.columns

train_f1, train_acc = evaluate_model(X_train_balanced.to_numpy(), y_train_balanced.to_numpy(), "Train (Balanced)", label_columns)
test_f1, test_acc = evaluate_model(X_test.to_numpy(), y_test.to_numpy(), "Test (Original)", label_columns)



Train (Balanced) Set Evaluation
Overall Sample-wise F1 Score:  0.532
Overall Sample-wise Accuracy: 0.4

Train (Balanced) Set - First 5 Labels:
is_student: F1 = 0.875, Acc = 0.949
is_disabled: F1 = 0.682, Acc = 0.935
is_female: F1 = 0.657, Acc = 0.914
is_male: F1 = 0.217, Acc = 0.968
is_transgender: F1 = 0.769, Acc = 0.995

Test (Original) Set Evaluation
Overall Sample-wise F1 Score:  0.469
Overall Sample-wise Accuracy: 0.308

Test (Original) Set - First 5 Labels:
is_student: F1 = 0.844, Acc = 0.935
is_disabled: F1 = 0.684, Acc = 0.933
is_female: F1 = 0.598, Acc = 0.906
is_male: F1 = 0.16, Acc = 0.972
is_transgender: F1 = 0.615, Acc = 0.993


In [38]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Initialize TF-IDF vectorizer (you can tune max_features as needed)
vectorizer = TfidfVectorizer(max_features=300)

# Fit and transform the 'Search_Keywords' column
tfidf_matrix = vectorizer.fit_transform(df['Search_Keywords'])

# Convert to DataFrame
df_tfidf = pd.DataFrame(tfidf_matrix.toarray(), columns=["kw_" + kw for kw in vectorizer.get_feature_names_out()])
df_tfidf.index = df.index  # Align index with original df

# Drop previous keyword features (if any)
X = X.drop(columns=[col for col in X.columns if col.startswith("kw_")], errors='ignore')

# Concatenate TF-IDF features with original feature set
X = pd.concat([X, df_tfidf], axis=1)


In [39]:
# 1. Split the updated data again 
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 2. Define base XGBoost model
xgb = XGBClassifier(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    use_label_encoder=False,
    eval_metric='logloss',
    random_state=42
)

# 3. Wrap with MultiOutputClassifier
clf2 = MultiOutputClassifier(xgb, n_jobs=1)

# 4. Fit the model
clf2.fit(X_train.to_numpy(), y_train.to_numpy())


MultiOutputClassifier(estimator=XGBClassifier(base_score=None, booster=None,
                                              callbacks=None,
                                              colsample_bylevel=None,
                                              colsample_bynode=None,
                                              colsample_bytree=0.8, device=None,
                                              early_stopping_rounds=None,
                                              enable_categorical=False,
                                              eval_metric='logloss',
                                              feature_types=None, gamma=None,
                                              grow_policy=None,
                                              importance_type=None,
                                              interaction_constraints=None,
                                              learning_rate=0.1, max_bin=None,
                                              max_cat_threshold=None,
                                              max_cat_to_onehot=None,
                                              max_delta_step=None, max_depth=5,
                                              max_leaves=None,
                                              min_child_weight=None,
                                              missing=nan,
                                              monotone_constraints=None,
                                              multi_strategy=None,
                                              n_estimators=100, n_jobs=None,
                                              num_parallel_tree=None,
                                              random_state=42, ...),
                      n_jobs=1)

In [40]:
train_f1, train_acc = evaluate_model(X_train.to_numpy(), y_train.to_numpy(), "Train", col_names=y.columns)
test_f1, test_acc = evaluate_model(X_test.to_numpy(), y_test.to_numpy(), "Test", col_names=y.columns)

ValueError: Feature shape mismatch, expected: 11153, got 304

In [19]:
!pip install iterative-stratification

  Obtaining dependency information for iterative-stratification from https://files.pythonhosted.org/packages/ba/01/d8d8d138713356ab4fecc5e69626c528e682c4ec2a596c707c2eb2d6351e/iterative_stratification-0.1.9-py3-none-any.whl.metadata


In [22]:
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit

In [23]:
# pip install iterstrat imbalanced-learn xgboost
import numpy as np
import pandas as pd
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.multioutput import MultiOutputClassifier

In [25]:
import numpy as np
import xgboost as xgb
from sklearn.metrics import f1_score, precision_recall_fscore_support
from tqdm import tqdm

# X_train, X_test, y_train, y_test are DataFrames (no target cols in X)
# float_cols already scaled and X contains final features (sparse/dense). 
# Ensure X_train, X_test are NumPy or sparse matrices as needed.

# Convert to DMatrix formats lazily when training each label
X_tr_mat = xgb.DMatrix(X_train.values, label=None)  # we'll set label per loop
X_te_mat = xgb.DMatrix(X_test.values)               # set label per loop for eval

params_base = {
    "objective": "binary:logistic",
    "eval_metric": "logloss",
    "max_depth": 5,
    "eta": 0.1,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "verbosity": 0,
    "seed": 42
}

n_rounds = 100
models = {}
y_pred_proba = np.zeros((X_test.shape[0], y_test.shape[1]))

for i, label in enumerate(y_test.columns):
    y_tr_label = y_train.iloc[:, i].values.astype(float)
    y_te_label = y_test.iloc[:, i].values.astype(float)

    # compute scale_pos_weight = (#negatives / #positives)
    pos = y_tr_label.sum()
    neg = len(y_tr_label) - pos
    if pos == 0:
        # no positive examples in train — skip training and predict zeros
        print(f"Label {label} has zero positives in train; predicting zeros.")
        y_pred_proba[:, i] = 0.0
        continue

    scale_pos_weight = float(neg / pos) if pos > 0 else 1.0

    params = params_base.copy()
    params["scale_pos_weight"] = scale_pos_weight

    dtrain = xgb.DMatrix(X_train.values, label=y_tr_label)
    dtest = xgb.DMatrix(X_test.values,  label=y_te_label)

    bst = xgb.train(params, dtrain, num_boost_round=n_rounds)
    models[label] = bst

    y_pred_proba[:, i] = bst.predict(dtest)

# Convert probabilities to binary predictions (default threshold 0.5; you can tune per-label)
y_pred = (y_pred_proba >= 0.5).astype(int)
y_pred_df = pd.DataFrame(y_pred, columns=y_test.columns, index=y_test.index)

# Evaluate overall (macro F1) and per-label metrics
macro_f1 = f1_score(y_test.values.flatten(), y_pred.flatten(), average='macro')
print("Macro F1 (flattened):", macro_f1)

# Per-label F1 / precision / recall
for i, label in enumerate(y_test.columns):
    p, r, f1, _ = precision_recall_fscore_support(y_test.iloc[:, i], y_pred_df.iloc[:, i], average='binary', zero_division=0)
    print(f"{label}: precision={p:.3f}, recall={r:.3f}, f1={f1:.3f}")


Macro F1 (flattened): 0.7624495254485043
is_student: precision=0.828, recall=0.864, f1=0.846
is_disabled: precision=0.598, recall=0.629, f1=0.613
is_female: precision=0.622, recall=0.595, f1=0.608
is_male: precision=0.132, recall=0.360, f1=0.194
is_transgender: precision=0.750, recall=0.600, f1=0.667
is_girl_child: precision=0.355, recall=0.537, f1=0.427
is_sc_st: precision=0.667, recall=0.602, f1=0.632
is_obc: precision=0.325, recall=0.641, f1=0.431
is_bpl: precision=0.592, recall=0.569, f1=0.580
is_farmer: precision=0.750, recall=0.857, f1=0.800
is_unemployed: precision=0.326, recall=0.583, f1=0.418
is_self_employed: precision=0.312, recall=0.636, f1=0.419
is_salaried_employee: precision=0.390, recall=0.471, f1=0.427
is_daily_wage_worker: precision=0.224, recall=0.647, f1=0.333
is_job_seeker: precision=0.750, recall=0.600, f1=0.667
is_worker: precision=0.824, recall=0.794, f1=0.809
is_labour: precision=0.763, recall=0.871, f1=0.813
is_school_student: precision=0.394, recall=0.812, f1

In [26]:
from sklearn.metrics import accuracy_score, f1_score

# Convert to numpy arrays
yt = y_test.values
yp = y_pred_df.values

# 1. Overall exact match accuracy (all labels correct for a row)
overall_accuracy = accuracy_score(yt, yp)

# 2. Macro F1 (treat each label equally)
macro_f1 = f1_score(yt, yp, average='macro')

# 3. Micro F1 (treat each instance of a label equally)
micro_f1 = f1_score(yt, yp, average='micro')

# 4. Weighted F1 (weighted by support of each label)
weighted_f1 = f1_score(yt, yp, average='weighted')

print("Overall Exact-Match Accuracy:", overall_accuracy)
print("Macro F1 Score:", macro_f1)
print("Micro F1 Score:", micro_f1)
print("Weighted F1 Score:", weighted_f1)


Overall Exact-Match Accuracy: 0.1659919028340081
Macro F1 Score: 0.38034097271965533
Micro F1 Score: 0.549663928304705
Weighted F1 Score: 0.5824108575895247
